In [1]:
import time
import pandas as pd
from google_play_scraper import reviews, Sort

APPS = {
    "netflix": "com.netflix.mediaclient",
    "prime":   "com.amazon.avod.thirdpartyclient",
}
LANG, COUNTRY = "en", "us"
STOP_BEFORE = pd.Timestamp("2022-01-01")   # newest first, so stop once we pass this date


def fetch_reviews(app_id, score, page_size=200, max_retries=5, pause=1.0):
    """Reviews of one app with one star rating, newest first, back to STOP_BEFORE.
    Retries a page if Google returns an error or an empty page."""
    all_rows = []
    batch, token = reviews(app_id, lang=LANG, country=COUNTRY, sort=Sort.NEWEST,
                           count=page_size, filter_score_with=score)
    all_rows.extend(batch)

    while token is not None and token.token is not None:
        if all_rows and all_rows[-1]["at"] < STOP_BEFORE:   # reached reviews older than 2022
            break

        for attempt in range(max_retries):
            try:
                batch, new_token = reviews(app_id, continuation_token=token)
                if batch:
                    break
            except Exception as e:
                print(f"  error: {e} (retry {attempt + 1}/{max_retries})")
            time.sleep(pause * (attempt + 1))
        else:
            print(f"  stopped: {max_retries} empty/failed pages in a row")
            break

        all_rows.extend(batch)
        token = new_token
        if len(all_rows) % 10000 < page_size:
            print(f"  {score}-star: {len(all_rows):,} so far (now at {all_rows[-1]['at']:%Y-%m})")

    return all_rows


for app_name, app_id in APPS.items():
    frames = []
    for score in [1, 2, 3, 4, 5]:
        print(f"{app_name}: fetching {score}-star reviews...")
        rows = fetch_reviews(app_id, score)
        print(f"  -> {len(rows):,} reviews")
        frames.append(pd.DataFrame(rows))

    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset="reviewId")
    df = df[pd.to_datetime(df["at"]) >= STOP_BEFORE]        # drop the few pre-2022 reviews from the last page
    df["app"] = app_name
    df["extracted_on"] = pd.Timestamp.today().date()

    df.to_csv(f"{app_name}_reviews_raw.csv", index=False)
    print(f"{app_name}: {len(df):,} reviews since 2022 saved")
    print(pd.to_datetime(df["at"]).dt.year.value_counts().sort_index(), "\n")

netflix: fetching 1-star reviews...
  1-star: 10,000 so far (now at 2026-04)
  1-star: 20,000 so far (now at 2025-09)
  1-star: 30,000 so far (now at 2025-01)
  1-star: 40,000 so far (now at 2024-09)
  1-star: 50,000 so far (now at 2024-04)
  1-star: 60,000 so far (now at 2023-12)
  1-star: 70,000 so far (now at 2023-07)
  1-star: 80,000 so far (now at 2023-04)
  1-star: 90,000 so far (now at 2023-04)
  1-star: 100,000 so far (now at 2023-01)
  1-star: 110,000 so far (now at 2022-08)
  1-star: 120,000 so far (now at 2022-05)
  1-star: 130,000 so far (now at 2022-01)
  -> 133,600 reviews
netflix: fetching 2-star reviews...
  2-star: 10,000 so far (now at 2023-08)
  -> 18,400 reviews
netflix: fetching 3-star reviews...
  3-star: 10,000 so far (now at 2023-09)
  3-star: 20,000 so far (now at 2022-02)
  -> 21,200 reviews
netflix: fetching 4-star reviews...
  4-star: 10,000 so far (now at 2024-06)
  4-star: 20,000 so far (now at 2022-10)
  4-star: 30,000 so far (now at 2022-01)
  -> 31,000 